# Classification Results
Results-only viewer — no training code. All models were trained via `src/models/` scripts.
Run cells top-to-bottom from the project root.

In [ ]:
import glob
import os

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
METRICS  = os.path.join(ROOT, 'outputs', 'metrics')
FIGURES  = os.path.join(ROOT, 'outputs', 'figures')

def latest(pattern):
    """Return the most recently modified file matching a glob pattern."""
    files = sorted(glob.glob(pattern), key=os.path.getmtime)
    if not files:
        raise FileNotFoundError(f'No file matching: {pattern}')
    return files[-1]

## 1. Class Distribution

In [ ]:
y_train = np.load(os.path.join(ROOT, 'data', 'processed', 'y_train.npy'))
le      = joblib.load(os.path.join(ROOT, 'models', 'label_encoder.joblib'))

counts = pd.Series(y_train).map(dict(enumerate(le.classes_))).value_counts().sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
counts.plot(kind='barh', ax=ax)
ax.set_xlabel('Number of training samples')
ax.set_title('Class distribution — training set')
fig.tight_layout()

out_path = os.path.join(FIGURES, 'class_distribution.png')
os.makedirs(FIGURES, exist_ok=True)
fig.savefig(out_path, dpi=150)
plt.show()
print(f'Saved → {out_path}')

## 2. Model Comparison

In [ ]:
lr_path  = latest(os.path.join(METRICS, '*lr*report*.csv'))
rf_path  = latest(os.path.join(METRICS, '*rf*report*.csv'))
xgb_path = latest(os.path.join(METRICS, '*xgb*report*.csv'))

print(f'LR  report: {os.path.basename(lr_path)}')
print(f'RF  report: {os.path.basename(rf_path)}')
print(f'XGB report: {os.path.basename(xgb_path)}')

aggregate_rows = {'accuracy', 'macro avg', 'weighted avg'}

def load_f1(path, col_name):
    df = pd.read_csv(path, index_col=0)
    return df.loc[~df.index.isin(aggregate_rows), 'f1-score'].rename(col_name)

comparison = pd.concat([
    load_f1(lr_path,  'LR F1'),
    load_f1(rf_path,  'RF F1'),
    load_f1(xgb_path, 'XGB F1'),
], axis=1).sort_values('XGB F1', ascending=False)

comparison.index.name = 'Department'

out_path = os.path.join(METRICS, 'model_comparison.csv')
comparison.to_csv(out_path)
print(f'\nSaved → {out_path}')
display(comparison.round(4))

## 3. Confusion Matrices

In [ ]:
for label, pattern in [
    ('Logistic Regression', os.path.join(FIGURES, '*lr*confusion*.png')),
    ('Random Forest',       os.path.join(FIGURES, '*rf*confusion*.png')),
    ('XGBoost',             os.path.join(FIGURES, '*xgb*confusion*.png')),
]:
    path = latest(pattern)
    print(f'{label} — {os.path.basename(path)}')
    display(Image(filename=path, width=800))

## 4. Feature Importance

In [ ]:
for label, pattern in [
    ('RF feature importance (mean decrease in impurity)', os.path.join(FIGURES, '*rf*feature*importance*.png')),
    ('SHAP global importance (mean |SHAP|)',              os.path.join(FIGURES, '*shap*global*.png')),
]:
    path = latest(pattern)
    print(f'{label} — {os.path.basename(path)}')
    display(Image(filename=path, width=700))

## 5. Key Findings

| | |
|---|---|
| **Best model** | ___ |
| **Macro F1** | ___ |

**Top 3 most predictable departments:** ___

**Top 3 hardest departments:** ___

**Most important features per SHAP:** ___

**One surprising finding:** ___